> **v2 (2026-09-15).** The first run used a fixed 300-epoch budget with no early
> stopping, and `trainer.train()` does not reload the best-epoch checkpoint at the end --
> so that run scored whatever state training happened to be in at epoch 300, not the best
> epoch found (which was earlier and better: geometry4 fold 1 best val_MAE 7.405 K at
> epoch 120 vs 7.746 K at epoch 300). This version fixes that (explicit best-checkpoint
> reload before scoring), replaces the fixed epoch count with `patience`-based early
> stopping so "more epochs" is tested honestly rather than guessed at, and adds
> `cno-fno` + attention (FiLM conditioning + CNO multiscale + axial self-attention +
> PDE/flux physics loss) -- the strongest architecture this repo implements -- run
> alongside plain FNO so an epoch/eval effect cannot be confused with an architecture
> effect. First-run result for reference: plain FNO at fixed 300 epochs was BELOW the
> linear-on-field ceiling on both geometries (geometry4 -0.722, geometry6 -0.378), and
> the Sec 9.15c gap prediction held only weakly (predicted gap 0.427, observed 0.082).

# geometry4-shelf vs geometry6-shelf: does an FNO inherit the field representation's difficulty?

**Sec 9.15c test.** Trains an FNO on both shelf-layout geometries under 5-fold CV and asks
which representation its accuracy tracks.

This notebook runs the one experiment `docs/report.md` Sec 9.15c makes a falsifiable
prediction about. It is **not** an architecture search and not an attempt to win anything.

## The prediction being tested

Sec 9.15c found that "linearly solvable" is a property of the **input representation**, not of
the dataset. On the *same* 45 geometry4-shelf files:

| representation | who gets it | linear spatial R2 |
|---|---|---|
| compact vector (block powers + positions + HTC + ambient, ~10 scalars) | `scripts/baselines.py` ridge | **-0.667** |
| full per-cell power field | the linearity audit, and an FNO | **0.962** |

geometry6-shelf is hard in **both** representations (0.536 from the field).

An FNO consumes the *field*. So:

> **Prediction.** FNO accuracy should track the difficulty of the task *in the field
> representation* (0.94 on geometry4 vs 0.51 on geometry6 -- a gap of ~0.43), **not** the
> difficulty in the compact representation ridge sees (-0.67 vs -2.87 -- a gap of ~2.2).

**Why the test is framed on absolute R2 and not on "margin over ridge".** A first draft of
this notebook tested whether the FNO's *margin over ridge* is larger on geometry4. That test
is broken: ridge scores -2.87 on geometry6 against -0.67 on geometry4, so **any** competent
model beats it by more on geometry6 mechanically, whatever the representation story is. The
margin is confounded by how bad the baseline is. What Sec 9.15c actually predicts is that the
FNO inherits the *field* representation's difficulty ordering, so that is what is tested:

| quantity | geometry4 | geometry6 | gap |
|---|---|---|---|
| ridge, compact representation (5-fold CV) | -0.674 | -2.866 | 2.19 |
| linear, field representation (5-fold CV) | **+0.941** | **+0.513** | **0.43** |
| FNO, field representation | ? | ? | ? |

If the FNO's gap is near **0.43**, it tracks the field representation -- Sec 9.15c holds. If it
is near **2.19**, it tracks the compact representation, which would be strange and would mean
something other than representation governs difficulty. If the FNO lands well *below* the
linear-on-field ceiling on both, the field representation is not usable at n=45 and the
interesting comparison is FNO vs linear-on-field, not FNO vs ridge.

**This can fail in three informative ways:**

1. FNO gap near 2.19 rather than 0.43 -> the representation story is wrong; difficulty is not
   representation-mediated.
2. FNO loses to ridge on both -> consistent with Sec 9.12d, where ridge beat both FNO configs
   on every hotspot metric under CV; would say the field representation is unusable at n=45.
3. FNO wins on both by similar margins -> representation matters but does not explain the
   geometry4/geometry6 difference.

Record whichever happens. A null result is a result, and Sec 9.15c should be narrowed if the
prediction fails.

## Protocol (fixed so results are comparable to Sec 9.15b)

- **5-fold CV over all 45 scenarios**, folds from `kfold_indices(n, 5, seed=0)` copied
  byte-for-byte from `scripts/hotspot_eval.py`, so the FNO is scored on the *same held-out
  fields* as the ridge numbers already in the report.
- **Detrended metrics** copied byte-for-byte from `scripts/layout_cv.py::per_scenario_stats`.
  Raw MAE is dominated by the ~325 K offset (Sec 9.1) and must not be used.
- **Identical epoch budget for both geometries.** Unequal budgets caused the convergence
  confound in Sec 9.12.
- Ridge is recomputed **in this notebook on the same folds**, not quoted, so the comparison
  cannot drift from a stale number.

## Data you must attach

**`src/` is cloned from GitHub** -- the repo is public and tracks it, so no upload is needed.
Set **Settings -> Internet -> On**. (If internet is off, the notebook falls back to an
attached dataset containing `src/`.)

**The data must still be attached.** `data/` is gitignored, so the `.npz` files are *not* in
the repo:

| dataset | contents | size |
|---|---|---|
| geometry4 shelf | 45 `.npz` from `data/3d-ice-layout-geometry4/geometry4/` | ~12 MB |
| geometry6 shelf | 45 `.npz` from `data/3d-ice-layout-geometry6/geometry6/` | ~31 MB |

Slugs do not matter -- both are located by searching attached datasets for their contents.
One combined dataset works as well as two.

Enable **GPU T4 x1**. Expect ~2-4 h; geometry6 is 2.5x the grid of geometry4
(56x168x15 = 141,120 cells vs 100x56x10 = 56,000).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)

import torch
print('Python :', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
    print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: no GPU. Enable Settings -> Accelerator -> GPU T4 x1 before training cells.')

In [ ]:
import sys, subprocess
from pathlib import Path

# src/ is cloned from GitHub (public repo, tracks src/) -- no Kaggle dataset needed;
# requires Settings -> Internet: On, falls back to an attached dataset if off.
# data/ is gitignored, so the .npz files (~43 MB) must still be attached separately.
REPO = 'https://github.com/rajul-kk/thermo-3dic-surrogates.git'
INPUT = Path('/kaggle/input')
OUT_DIR = Path('/kaggle/working/checkpoints/layout_repr_test')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Attached datasets under /kaggle/input:')
if INPUT.exists() and any(INPUT.iterdir()):
    for d in sorted(INPUT.iterdir()):
        n_npz = len(list(d.rglob('*.npz')))
        has_src = bool(list(d.rglob('src/fno/model.py')))
        tag = []
        if has_src:
            tag.append('HAS src/')
        if n_npz:
            tag.append(str(n_npz) + ' .npz')
        print('  {:<42} {}'.format(d.name, ' | '.join(tag) if tag else '(neither src/ nor .npz)'))
else:
    print('  (nothing attached)')

# --- src/: clone from GitHub, else fall back to an attached dataset ----------
SRC_ROOT = None
clone_dir = Path('/kaggle/working/repo')
# /kaggle/working persists across re-runs, so a stale clone could linger after a
# src/ fix; always sync to the latest commit instead of cloning only if missing.
if clone_dir.exists():
    r = subprocess.run(['git', '-C', str(clone_dir), 'fetch', '--depth', '1', 'origin', 'main'],
                       capture_output=True, text=True)
    if r.returncode == 0:
        r = subprocess.run(['git', '-C', str(clone_dir), 'reset', '--hard', 'origin/main'],
                           capture_output=True, text=True)
    if r.returncode != 0:
        print('\ngit sync of existing clone failed (this is expected if Internet is Off):')
        print('  ' + (r.stderr.strip().splitlines() or ['?'])[-1])
        print('Deleting the stale clone and retrying fresh...')
        import shutil
        shutil.rmtree(clone_dir, ignore_errors=True)
if not clone_dir.exists():
    r = subprocess.run(['git', 'clone', '--depth', '1', REPO, str(clone_dir)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print('\ngit clone failed (this is expected if Settings -> Internet is Off):')
        print('  ' + (r.stderr.strip().splitlines() or ['?'])[-1])
if (clone_dir / 'src' / 'fno' / 'model.py').exists():
    SRC_ROOT = clone_dir
    sha = subprocess.run(['git', '-C', str(clone_dir), 'rev-parse', '--short', 'HEAD'],
                         capture_output=True, text=True).stdout.strip()
    print('\nsrc/ cloned from GitHub at commit ' + sha)
else:
    for hit in INPUT.rglob('src/fno/model.py'):
        SRC_ROOT = hit.parent.parent.parent
        print('\nsrc/ taken from attached dataset: ' + SRC_ROOT.name)
        break

assert SRC_ROOT is not None, (
    '\n\nCould not obtain src/.\n'
    'Easiest fix: Settings -> Internet -> On, then re-run (the repo is public and is\n'
    'cloned automatically).\n'
    'Offline alternative: zip the project src/ folder so the zip root CONTAINS src/\n'
    '(check src/fno/model.py is inside), upload as a Kaggle dataset and attach it.'
)
sys.path.insert(0, str(SRC_ROOT))

# --- data: must be attached; located by content, not by slug -----------------
def find_geom_root(geom):
    best, best_n = None, 0
    for d in (INPUT.iterdir() if INPUT.exists() else []):
        n = len(list(d.rglob(geom + '_*.npz')))
        if n > best_n:
            best, best_n = d, n
    return best, best_n

FILES = {}
for geom in ['geometry4', 'geometry6']:
    root, n = find_geom_root(geom)
    assert root is not None and n > 0, (
        '\n\nNo attached dataset contains ' + geom + '_*.npz.\n'
        'data/ is gitignored, so these are NOT in the GitHub repo and must be uploaded.\n'
        'Upload the 45 .npz files from data/3d-ice-layout-' + geom + '/' + geom + '/ as a\n'
        'Kaggle dataset and attach it (~12 MB for geometry4, ~31 MB for geometry6).\n'
        'The slug does not matter. See the list printed above for what is attached.'
    )
    FILES[geom] = sorted(root.rglob(geom + '_*.npz'))
    flag = '' if len(FILES[geom]) == 45 else '   <-- expected 45!'
    print('{}: {} scenarios from {}{}'.format(geom, len(FILES[geom]), root.name, flag))

In [ ]:
import logging, time, json
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.core.geometry_builders import get_geometry_by_name
from src.pinn.data_loader import compute_norm_stats
from src.fno.model import build_fno, build_cno_fno
from src.fno.ltfno import build_lt_fno
from src.fno.data_loader import FNODataset, predict_to_flat
from src.fno.trainer import FNOTrainer

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)-7s %(name)s: %(message)s',
                    datefmt='%H:%M:%S')
print('Imports OK')

In [ ]:
# Folds and metrics copied byte-for-byte from the repo so these numbers are directly
# comparable to docs/report.md Sec 9.15b -- do not "improve" them.

def kfold_indices(n, folds, seed):
    '''scripts/hotspot_eval.py::kfold_indices'''
    idx = np.arange(n)
    np.random.default_rng(seed).shuffle(idx)
    return [np.asarray(part) for part in np.array_split(idx, folds)]


def per_scenario_stats(y, p):
    '''scripts/layout_cv.py::per_scenario_stats -- detrended, spatial structure only.'''
    dy, dp = y - y.mean(), p - p.mean()
    ss = float(np.sum(dy ** 2))
    return {'det_mae': float(np.mean(np.abs(dy - dp))),
            'sigma': float(dy.std()),
            'r2': float(1.0 - np.sum((dy - dp) ** 2) / ss) if ss > 0 else float('nan'),
            'corr': float(np.corrcoef(dp, dy)[0, 1]) if dp.std() > 0 and dy.std() > 0 else 0.0}


def summarise(rows, label):
    r2 = np.array([r['r2'] for r in rows])
    dm = float(np.mean([r['det_mae'] for r in rows]))
    sig = float(np.mean([r['sigma'] for r in rows]))
    return {'model': label, 'n': len(rows), 'det_mae': dm, 'sigma': sig,
            'norm_err': dm / sig, 'r2_mean': float(r2.mean()),
            'r2_median': float(np.median(r2)), 'n_negative_r2': int((r2 < 0).sum()),
            'corr': float(np.mean([r['corr'] for r in rows]))}

FOLDS, SEED = 5, 0
print(f'{FOLDS}-fold CV, seed {SEED} -- identical folds to the report')

In [ ]:
# Ridge on the COMPACT representation, recomputed rather than quoted -- mirrors
# scripts/baselines.py (block powers/positions + HTC + ambient + TSV density).

def load_scenario(path):
    d = np.load(path, allow_pickle=True)
    return {'name': path.stem, 'temp': d['temp'].astype(np.float64),
            'power': d['power'].astype(np.float64),
            'meta': dict(d['metadata'][0])}


def collect_keys(scen, prefix):
    keys = set()
    for s in scen:
        keys |= {k for k in s['meta'] if k.startswith(prefix)}
    return sorted(keys)


def feature_vector(meta, block_keys, pos_keys):
    htc = float(meta.get('htc', 0.0))
    f = [float(meta.get(k, 0.0)) for k in block_keys]
    f += [float(meta.get(k, 0.0)) for k in pos_keys]
    f += [htc, 1.0 / htc if htc > 0 else 0.0,
          float(meta.get('t_ambient_kelvin', 298.15)),
          float(meta.get('tsv_density', 0.0))]
    return np.asarray(f, dtype=np.float64)


def ridge_cv(scen, lam=1.0):
    bk = collect_keys(scen, 'block_power_')
    if not bk:
        bk = [k for k in sorted(scen[0]['meta']) if k.startswith('block_') and 'power' in k]
    pk = collect_keys(scen, 'block_x_') + collect_keys(scen, 'block_y_')
    X = np.stack([feature_vector(s['meta'], bk, pk) for s in scen])
    Y = np.stack([s['temp'] for s in scen])
    rows = []
    for fold in kfold_indices(len(scen), FOLDS, SEED):
        te = set(fold.tolist())
        tr = np.array([i for i in range(len(scen)) if i not in te])
        Xtr, Ytr = X[tr], Y[tr]
        keep = Xtr.std(0) > 1e-12                      # drop constant columns
        mu, sd = Xtr[:, keep].mean(0), Xtr[:, keep].std(0) + 1e-12
        A = np.hstack([(Xtr[:, keep] - mu) / sd, np.ones((len(tr), 1))])
        W = np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ Ytr)
        B = np.hstack([(X[fold][:, keep] - mu) / sd, np.ones((len(fold), 1))])
        P = B @ W
        for j, i in enumerate(fold):
            rows.append(per_scenario_stats(Y[i], P[j]))
    return rows, int(keep.sum())

SCEN = {g: [load_scenario(p) for p in FILES[g]] for g in FILES}
RIDGE = {}
for g in SCEN:
    rows, nfeat = ridge_cv(SCEN[g])
    RIDGE[g] = summarise(rows, 'ridge (compact)')
    print(f'{g}: ridge over {nfeat} non-constant compact features -> '
          f"R2 mean {RIDGE[g]['r2_mean']:+.3f}  median {RIDGE[g]['r2_median']:+.3f}  "
          f"norm_err {RIDGE[g]['norm_err']:.3f}  R2<0 {RIDGE[g]['n_negative_r2']}/45")
print()
print('SANITY CHECK vs docs/report.md Sec 9.15b: geometry4 R2 mean should be near -0.667,')
print('geometry6 near -2.866 (median near -0.303). A large deviation means the folds or')
print('features differ from the report -- fix that before reading anything else here.')

In [ ]:
# Linear fit on the FIELD representation -- the same input the FNO gets, so a win
# is not ambiguous between "the network helped" and "the representation helped".

def linear_field_cv(scen, pca_k=8, lam=1e-2):
    X = np.stack([s['power'] for s in scen])
    Y = np.stack([s['temp'] for s in scen])
    rows = []
    for fold in kfold_indices(len(scen), FOLDS, SEED):
        te = set(fold.tolist())
        tr = np.array([i for i in range(len(scen)) if i not in te])
        Xtr = X[tr]
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-12
        Xc = (Xtr - mu) / sd
        # economy SVD of the (n_train x n_cells) matrix; n_train is 36, so cheap
        _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
        k = min(pca_k, Vt.shape[0])
        B = Vt[:k].T
        A = np.hstack([Xc @ B, np.ones((len(tr), 1))])
        W = np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ Y[tr])
        F = np.hstack([((X[fold] - mu) / sd) @ B, np.ones((len(fold), 1))])
        P = F @ W
        for j, i in enumerate(fold):
            rows.append(per_scenario_stats(Y[i], P[j]))
    return rows

LINFIELD = {}
for g in SCEN:
    LINFIELD[g] = summarise(linear_field_cv(SCEN[g]), 'linear (field)')
    print(f"{g}: linear-on-field -> R2 mean {LINFIELD[g]['r2_mean']:+.3f}  "
          f"median {LINFIELD[g]['r2_median']:+.3f}  norm_err {LINFIELD[g]['norm_err']:.3f}")

In [ ]:
# FNO config, identical across variants/geometries except grid shape. Epoch budget is
# patience-based early stopping (not a fixed count), and the best checkpoint is
# explicitly reloaded before scoring rather than trusting wherever training stopped.
CHANNELS    = 32
N_BLOCKS    = 4
MODES       = (16, 16, 8)
BATCH_SIZE  = 2          # geometry6 is 141k cells/scenario; keep VRAM headroom
LR          = 1e-3

EPOCHS_MAX   = 600       # hard cap; early stopping usually ends well before this
LOG_INTERVAL = 10        # a "check" happens every this many epochs
PATIENCE     = 8         # stop after this many checks (80 epochs) with no improvement

# cno-fno-attn only: FiLM + CNO multiscale + axial self-attention + physics loss,
# same PDE/flux weights as the A3/A3b/geometry7 notebooks for comparability.
N_CNO_LAYERS = 2
N_HEADS      = 4         # must divide CHANNELS evenly
PDE_WEIGHT   = 0.0      # v5: the FD residual assumes uniform z; 3D-ICE z-nodes are not
FLUX_WEIGHT  = 0.0      # uniform (Sec 9.24), so the physics loss is off until rebuilt on the FV operator

# 'fno' is rerun here (not quoted from the first pass) under the SAME epoch policy
# as 'cno-fno-attn' -- the strongest architecture this repo implements.
VARIANTS = ['fno', 'lt-fno', 'cno-fno-attn']   # lt-fno: report Sec 9.27

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    print('WARNING: DEVICE is CPU. Each epoch on this project\'s grids takes ~15-20x')
    print('longer on CPU than on a T4 -- this cell will silently let you run for hours.')
    print('Stop now and check Settings -> Accelerator -> GPU T4 x1, then restart the')
    print('session (not just re-run cells) before continuing.')

GRID = {}
for g in FILES:
    c = np.load(FILES[g][0], allow_pickle=True)['coords']
    GRID[g] = tuple(len(np.unique(c[:, i])) for i in (1, 0, 2))   # (length, width, z) = points_to_grid order
    print(f'{g}: grid {GRID[g]} = {int(np.prod(GRID[g])):,} cells')
print()
print(f'Device {DEVICE} | channels {CHANNELS} | blocks {N_BLOCKS} | modes {MODES}')
print(f'Epoch policy: up to {EPOCHS_MAX}, early-stop after {PATIENCE} checks '
      f'({PATIENCE * LOG_INTERVAL} epochs) with no improvement')
print(f'Variants: {VARIANTS}')

In [ ]:
def build_model(variant, grid):
    if variant == 'fno':
        return build_fno(grid, modes=MODES, hidden_ch=CHANNELS, n_blocks=N_BLOCKS,
                         device=DEVICE)
    if variant == 'cno-fno-attn':
        return build_cno_fno(grid, ch=CHANNELS, n_fno_blocks=N_BLOCKS,
                             n_cno_layers=N_CNO_LAYERS, use_attention=True,
                             n_heads=N_HEADS, device=DEVICE)
    if variant == 'lt-fno':
        return build_lt_fno(grid, modes=MODES, hidden_ch=CHANNELS, n_blocks=N_BLOCKS, device=DEVICE)
    raise ValueError(f'unknown variant {variant!r}')


def train_variant_cv(geom, variant):
    '''5-fold CV, fresh model per fold. Reloads the best-epoch checkpoint before scoring --
    trainer.train() leaves self.model PATIENCE checks past the best, a worse checkpoint.
       '''
    files = FILES[geom]
    grid = GRID[geom]
    # PI loss only for cno-fno-attn. grid_spacings depend only on the fixed geometry
    # type, not per-scenario chiplet placement, so the nominal object is valid here.
    use_pi = variant == 'cno-fno-attn'
    geometry_obj = get_geometry_by_name(geom) if use_pi else None
    pde_w = PDE_WEIGHT if use_pi else 0.0
    flux_w = FLUX_WEIGHT if use_pi else 0.0

    rows, times, best_epochs = [], [], []

    for fi, fold in enumerate(kfold_indices(len(files), FOLDS, SEED), 1):
        te = set(fold.tolist())
        tr_files = [files[i] for i in range(len(files)) if i not in te]
        te_files = [files[i] for i in fold]
        # Early stopping and best-checkpoint selection use scenarios held out of the TRAINING
        # fold. Before 2026-09-25 this passed te_ds, so model selection saw the test fold.
        rng = np.random.default_rng(SEED + fi)
        val_idx = set(rng.choice(len(tr_files), size=max(3, len(tr_files) // 9), replace=False).tolist())
        val_files = [f for k, f in enumerate(tr_files) if k in val_idx]
        tr_files = [f for k, f in enumerate(tr_files) if k not in val_idx]

        norm = compute_norm_stats(tr_files, {geom: get_geometry_by_name(geom)})
        tr_ds = FNODataset(tr_files, norm, grid)
        val_ds = FNODataset(val_files, norm, grid)
        te_ds = FNODataset(te_files, norm, grid)

        model = build_model(variant, grid)
        trainer = FNOTrainer(model, norm, tr_ds, val_ds,
                             output_dir=OUT_DIR / f'{geom}_{variant}_fold{fi}',
                             batch_size=BATCH_SIZE, epochs=EPOCHS_MAX, lr=LR,
                             device=DEVICE, use_amp=False,
                             log_interval=LOG_INTERVAL, patience=PATIENCE,
                             pde_weight=pde_w, flux_weight=flux_w, geometry=geometry_obj,
                             geometry_name=f'{geom}_{variant}_fold{fi}')
        t0 = time.time()
        ckpt_path = trainer.train()
        elapsed = time.time() - t0
        times.append(elapsed)
        best_epochs.append(len(trainer.history['epoch']) * LOG_INTERVAL)

        # Reload the BEST checkpoint -- see the docstring above for why this matters.
        state = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(state['model_state'])

        for j in range(len(te_ds)):
            pred, true = predict_to_flat(model, te_ds.items[j], DEVICE, norm)
            rows.append(per_scenario_stats(true, pred))
        print(f'  {geom}/{variant} fold {fi}/{FOLDS}: best val_MAE={state["val_mae_K"]:.3f} K '
              f'at ~epoch {state["epoch"]}, ran {best_epochs[-1]} epochs total, '
              f'{elapsed/60:.1f} min ({len(te_ds)} held-out fields)')
        del model, trainer
        torch.cuda.empty_cache()

    s = summarise(rows, variant)
    s['train_seconds_total'] = float(np.sum(times))
    s['train_seconds_per_fold'] = float(np.mean(times))
    s['epochs_per_fold'] = best_epochs
    return s


FNO = {v: {} for v in VARIANTS}
for variant in VARIANTS:
    for g in ['geometry4', 'geometry6']:          # geometry4 first: cheaper, fails faster
        print(f'=== {variant} on {g}-shelf ===')
        FNO[variant][g] = train_variant_cv(g, variant)
        r = FNO[variant][g]
        print(f"{g}/{variant}: R2 mean {r['r2_mean']:+.3f}  median {r['r2_median']:+.3f}  "
              f"norm_err {r['norm_err']:.3f}  R2<0 {r['n_negative_r2']}/45  "
              f"({r['train_seconds_total']/3600:.2f} GPU-h total, "
              f"epochs/fold {r['epochs_per_fold']})")
        print()
        # saved after every run so a session timeout still leaves the finished results
        (OUT_DIR / 'fno_results.json').write_text(json.dumps(FNO, indent=1))

In [ ]:
# Verdict, extended to every variant. Sec 9.15c predicts a FIELD-representation model
# inherits its difficulty gap (~0.427), not the compact one's (~2.192). Tested on
# absolute R2, not margin over ridge -- a margin test is confounded here.
import pandas as pd

recs = []
for g in ['geometry4', 'geometry6']:
    for s in (RIDGE[g], LINFIELD[g]):
        recs.append({'geometry': g, 'model': s['model'], 'det_mae': s['det_mae'],
                     'norm_err': s['norm_err'], 'r2_mean': s['r2_mean'],
                     'r2_median': s['r2_median'], 'n_negative_r2': s['n_negative_r2'],
                     'corr': s['corr']})
    for variant in VARIANTS:
        s = FNO[variant][g]
        recs.append({'geometry': g, 'model': f'{variant} (field)', 'det_mae': s['det_mae'],
                     'norm_err': s['norm_err'], 'r2_mean': s['r2_mean'],
                     'r2_median': s['r2_median'], 'n_negative_r2': s['n_negative_r2'],
                     'corr': s['corr']})
print(pd.DataFrame(recs).to_string(index=False, float_format=lambda v: f'{v:.4f}'))

gap = {'ridge_compact': RIDGE['geometry4']['r2_mean'] - RIDGE['geometry6']['r2_mean'],
       'linear_field': LINFIELD['geometry4']['r2_mean'] - LINFIELD['geometry6']['r2_mean']}
for variant in VARIANTS:
    gap[variant] = FNO[variant]['geometry4']['r2_mean'] - FNO[variant]['geometry6']['r2_mean']

print()
print('=' * 78)
print('geometry4 minus geometry6, detrended spatial R2 (mean)')
print('=' * 78)
print(f"  ridge, compact representation : {gap['ridge_compact']:+.3f}")
print(f"  linear, field representation  : {gap['linear_field']:+.3f}   <- Sec 9.15c predicts near this")
for variant in VARIANTS:
    d_field = abs(gap[variant] - gap['linear_field'])
    d_compact = abs(gap[variant] - gap['ridge_compact'])
    tracks = 'FIELD' if d_field < d_compact else 'COMPACT'
    print(f"  {variant:<16} (field)        : {gap[variant]:+.3f}   tracks {tracks}  "
          f"(|d_field|={d_field:.3f} vs |d_compact|={d_compact:.3f})")

print()
print('Does each variant beat the best LINEAR use of the SAME input (the field)?')
print('This is the headline regardless of the gap test: if a model with a strictly')
print('richer hypothesis class than a linear fit still loses to that linear fit on')
print('its own input, the extra capacity bought nothing on this data.')
for variant in VARIANTS:
    for g in ['geometry4', 'geometry6']:
        m = FNO[variant][g]['r2_mean'] - LINFIELD[g]['r2_mean']
        print(f'  {variant:<16} {g}: {m:+.3f} ({"ABOVE" if m > 0 else "below"} the '
              f'linear-on-field ceiling)')

out = {'folds': FOLDS, 'seed': SEED, 'epochs_max': EPOCHS_MAX, 'patience': PATIENCE,
       'log_interval': LOG_INTERVAL,
       'config': {'channels': CHANNELS, 'blocks': N_BLOCKS, 'modes': list(MODES),
                  'batch_size': BATCH_SIZE, 'lr': LR, 'n_cno_layers': N_CNO_LAYERS,
                  'n_heads': N_HEADS, 'pde_weight': PDE_WEIGHT, 'flux_weight': FLUX_WEIGHT},
       'grid': {g: list(GRID[g]) for g in GRID},
       'ridge_compact': RIDGE, 'linear_field': LINFIELD, 'fno': FNO,
       'geometry4_minus_geometry6_gap': gap}
(OUT_DIR / 'verdict_v2.json').write_text(json.dumps(out, indent=2))
print()
print(f'Saved {OUT_DIR / "verdict_v2.json"}')

In [ ]:
# R2 distributions across every model tested. The mean is heavy-tailed on these
# datasets (Sec 9.15b), so the median matters as much as the centre -- both shown.
n_models = 2 + len(VARIANTS)
colors = ['#bb2233', '#ee9933'] + ['#2277aa', '#22aa77', '#8833cc'][:len(VARIANTS)]
labels = ['ridge\n(compact)', 'linear\n(field)'] + [v.replace('-', '\n') for v in VARIANTS]

fig, axes = plt.subplots(1, 2, figsize=(6 + 2.2 * n_models, 4.5), sharey=True)
for ax, g in zip(axes, ['geometry4', 'geometry6']):
    means = [RIDGE[g]['r2_mean'], LINFIELD[g]['r2_mean']] + [FNO[v][g]['r2_mean'] for v in VARIANTS]
    medians = [RIDGE[g]['r2_median'], LINFIELD[g]['r2_median']] + [FNO[v][g]['r2_median'] for v in VARIANTS]
    ax.bar(range(n_models), means, color=colors)
    ax.errorbar(range(n_models), medians, fmt='k_', markersize=22, linestyle='none', label='median')
    ax.set_xticks(range(n_models))
    ax.set_xticklabels(labels)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_title(f'{g}-shelf  (bar = mean, tick = median)')
    ax.set_ylabel('detrended spatial R$^2$')
    ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'r2_comparison_v2.png', dpi=130)
plt.show()

print('Training cost (T4):')
for variant in VARIANTS:
    for g in FNO[variant]:
        r = FNO[variant][g]
        print(f"  {variant}/{g}: {r['train_seconds_total']/3600:.2f} GPU-h total, "
              f"{r['train_seconds_per_fold']/60:.1f} min/fold avg, "
              f"epochs/fold {r['epochs_per_fold']}")

## Reading the result honestly

**Report whatever comes out, including a null or negative result.** Three things to resist:

1. **Do not tune until the prediction passes.** If it fails at 300 epochs, that is the result
   at 300 epochs. Re-running with a bigger model until Sec 9.15c is confirmed is precisely the
   researcher-degrees-of-freedom failure this project documents (`docs/report.md` Sec 9.1,
   Sec 9.12; McGreivy & Hakim 2024 on outcome-reporting bias).
2. **Check the ridge sanity line first.** If in-notebook ridge does not land near Sec 9.15b's
   -0.667 / -2.866, the folds or features differ and *no* comparison here is valid.
3. **Do not read "margin over ridge" as the result.** Ridge scores -2.87 on geometry6 and
   -0.67 on geometry4, so a larger margin on geometry6 is mechanical and says nothing about
   representation. The gap test above exists because of that; an earlier draft of this
   notebook got it wrong.

**If the prediction holds**, keep the claim narrow: *on these datasets, at this budget, the
benefit of a neural operator tracks how impoverished the baseline's representation is rather
than how hard the task is.* That is a statement about benchmark design, not about FNO quality.

**Limits to state alongside any result:**

- n=45 per geometry. CV uses every scenario as held-out once, which removes the single-split
  lottery of Sec 9.15 but not the sparsity.
- One architecture, one budget, **one seed per fold**. Sec 9.12d already showed a 5-scenario
  split inverting an FNO-vs-ridge ranking, so this is the weakest part of the design: if the
  two margins come out close, run 3 seeds per fold before claiming anything.
- geometry4 and geometry6 differ in grid size (56k vs 141k cells) as well as layout
  difficulty, so they are not perfectly matched. A margin difference could partly reflect
  that, and the write-up must say so.

## What this version adds, and what to read from it

**Two effects are isolated, not bundled.** `fno` is rerun here under the same
early-stopping policy as `cno-fno-attn`, so a difference between the two variants is
an architecture effect, and a difference between this `fno` row and the first run's
fixed-300-epoch `fno` row is an epoch-budget-and-eval-correctness effect. Read both
comparisons; don't collapse them into "the new run is better/worse".

**If `cno-fno-attn` still loses to `linear (field)` on both geometries**, that is a
materially stronger result than the first run's finding, because it rules out "the
network just needed more epochs" and "the network just needed more capacity" as the
explanation -- the strongest architecture this repo implements, trained to convergence
via early stopping, still doesn't beat a closed-form fit on the same input. Report
that plainly rather than reaching for a third variant to try next; at some point
"try another architecture" becomes the researcher-degrees-of-freedom problem this
project has documented repeatedly (Sec 9.1, Sec 9.12, McGreivy & Hakim 2024).

**If `cno-fno-attn` beats the linear-on-field ceiling on geometry6 but not geometry4**,
that would be a genuinely interesting result worth writing up carefully: it would say
the extra capacity earns its cost specifically where the task is intrinsically hard
(geometry6, low effective DOF in the field representation per Sec 9.16a) rather than
where ridge is merely handicapped by a poor representation (geometry4). Check this
explicitly rather than only reading off the gap test.

**Compute cost caveat.** `cno-fno-attn` does an extra finite-difference PDE-residual
forward pass per batch (the physics loss), so each epoch costs more than plain FNO's.
Early stopping bounds this, but if PATIENCE is too generous relative to how noisy
val_MAE is on 9 held-out scenarios, a fold can run needlessly long. If a fold is
clearly not improving after a few hundred epochs, it is reasonable to lower
EPOCHS_MAX or PATIENCE and note that in the write-up rather than let one fold burn
the session's GPU-hour budget.